In [24]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.prompts import (ChatPromptTemplate,
                                    SystemMessagePromptTemplate,
                                    HumanMessagePromptTemplate,
                                    MessagesPlaceholder)

# Prompt Template

## ChatPromptTemplate

In [1]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个有帮助的AI机器人，你的名字是{name}。"),
        ("user", "你好，最近怎么样？"),
        ("ai", "我很好，谢谢！"),
        ("user", "{user_input}"),
    ]
)

prompt_value = prompt_template.invoke(
    {
        "name": "Aiden",
        "user_input": "1 * 0 = ?"
    }
)

print(prompt_value)
print(type(prompt_value))

messages=[SystemMessage(content='你是一个有帮助的AI机器人，你的名字是Aiden。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，最近怎么样？', additional_kwargs={}, response_metadata={}), AIMessage(content='我很好，谢谢！', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='1 * 0 = ?', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>


## 模板调用的3种方式

In [2]:
# Error Case (Validation)
prompt_template.invoke({"test": "demo"})

KeyError: "Input to ChatPromptTemplate is missing variables {'name', 'user_input'}.  Expected: ['name', 'user_input'] Received: ['test']\nNote: if you intended {name} to be part of the string and not a variable, please escape it with double curly braces like: '{{name}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "

In [3]:
prompt_str = prompt_template.format(name="Aiden", user_input="1 * 0 = ?")
print(prompt_str)
print(type(prompt_str))

System: 你是一个有帮助的AI机器人，你的名字是Aiden。
Human: 你好，最近怎么样？
AI: 我很好，谢谢！
Human: 1 * 0 = ?
<class 'str'>


In [4]:
prompt_message = prompt_template.format_messages(name="Aiden", user_input="1 * 0 = ?")
print(prompt_message)
print(type(prompt_message))

[SystemMessage(content='你是一个有帮助的AI机器人，你的名字是Aiden。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，最近怎么样？', additional_kwargs={}, response_metadata={}), AIMessage(content='我很好，谢谢！', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='1 * 0 = ?', additional_kwargs={}, response_metadata={})]
<class 'list'>


## LLM invoke

In [8]:
load_dotenv(override=True)

model = init_chat_model(model="deepseek:deepseek-v4-flash")

response = model.invoke(prompt_value)
print(response.content_blocks)

response = model.invoke(prompt_str)
print(response.content_blocks)

response = model.invoke(prompt_message)
print(response.content_blocks)

[{'type': 'reasoning', 'reasoning': '我们被问到1 * 0 = ?。显然答案是0。但需要以中文回复。'}, {'type': 'text', 'text': '1 * 0 = 0。'}]
[{'type': 'reasoning', 'reasoning': '我们被问到"1 * 0 = ?"。这是一个简单的数学问题。答案是0。所以回复应该是"0"。但注意对话历史：人类说"1 * 0 = ?"，AI应该直接回答。因为系统说AI名字是Aiden，但这里不需要特别称呼。直接回答即可。'}, {'type': 'text', 'text': '0'}]
[{'type': 'reasoning', 'reasoning': '我们正在计算1乘以0等于多少。这是一个简单的乘法问题。任何数乘以0都等于0。所以答案是0。'}, {'type': 'text', 'text': '1 × 0 = 0。'}]


## 更丰富的初始化参数类型

### dict列表类型

In [9]:
prompt_template_from_dict = ChatPromptTemplate.from_messages(
    [
        {
            "role": "system",
            "content": "你的名字是{role}."
        },
        {
            "role": "user",
            "content": "2 * 3 = ?"
        }
    ]
)
print(prompt_template_from_dict.invoke({"role": "assistant"}))

messages=[SystemMessage(content='你的名字是assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='2 * 3 = ?', additional_kwargs={}, response_metadata={})]


In [ ]:
### Message列表类型

In [10]:
from langchain_core.messages import SystemMessage, HumanMessage

chat_prompt_template = ChatPromptTemplate.from_messages([
    SystemMessage(content="我是一个贴心的智能助手"),
    HumanMessage(content="我的问题是:{word}英文怎么说？")
])
messages = chat_prompt_template.invoke({"word": "人工智能"})
print(messages)

messages=[SystemMessage(content='我是一个贴心的智能助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是:{word}英文怎么说？', additional_kwargs={}, response_metadata={})]


### MessagePromptTemplate列表类型

In [16]:
system_message_prompt_template = SystemMessagePromptTemplate.from_template("你是一个 {role}")
human_message_prompt_template = HumanMessagePromptTemplate.from_template("给我解释 {concept}，用浅显易懂的语言")

chat_prompt_template = ChatPromptTemplate.from_messages([
    system_message_prompt_template,
    human_message_prompt_template
])

prompt_values = chat_prompt_template.invoke({"role": "assistant", "concept": "RBMK"})
print(prompt_values)

messages=[SystemMessage(content='你是一个 assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='给我解释 RBMK，用浅显易懂的语言', additional_kwargs={}, response_metadata={})]


BaseChatPromptTemplate列表类型

In [17]:
nested_prompt_template1 = ChatPromptTemplate.from_messages([("system", "我是一个人工智能助手，我的名字叫{name}")])
nested_prompt_template2 = ChatPromptTemplate.from_messages([("human", "很高兴认识你,我的问题是{question}")])
prompt_template = ChatPromptTemplate.from_messages([
    nested_prompt_template1, nested_prompt_template2
])
prompt_template.invoke({"name": "小智", "question": "你为什么这么帅？"})

ChatPromptValue(messages=[SystemMessage(content='我是一个人工智能助手，我的名字叫小智', additional_kwargs={}, response_metadata={}), HumanMessage(content='很高兴认识你,我的问题是你为什么这么帅？', additional_kwargs={}, response_metadata={})])

In [18]:
# 示例 1: 使用 BaseMessage（已实例化的消息）
system_msg = SystemMessage(content="你是一个AI工程师。")
human_msg = HumanMessage(content="你好！")
# 示例 2: 使用 BaseMessagePromptTemplate
system_prompt = SystemMessagePromptTemplate.from_template("你是一个{role}.")
human_prompt = HumanMessagePromptTemplate.from_template("{user_input}")
# 示例 3: 使用 BaseChatPromptTemplate（嵌套的 ChatPromptTemplate）
nested_prompt = ChatPromptTemplate.from_messages([("system", "嵌套提示词")])
prompt = ChatPromptTemplate.from_messages([
    system_msg,  # MessageLike (BaseMessage)
    human_msg,  # MessageLike (BaseMessage)
    system_prompt,  # MessageLike (BaseMessagePromptTemplate)
    human_prompt,  # MessageLike (BaseMessagePromptTemplate)
    nested_prompt,  # MessageLike (BaseChatPromptTemplate)
])
prompt.invoke({"role": "人工智能专家", "user_input": "介绍一下大模型的应用场景"})

ChatPromptValue(messages=[SystemMessage(content='你是一个AI工程师。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好！', additional_kwargs={}, response_metadata={}), SystemMessage(content='你是一个人工智能专家.', additional_kwargs={}, response_metadata={}), HumanMessage(content='介绍一下大模型的应用场景', additional_kwargs={}, response_metadata={}), SystemMessage(content='嵌套提示词', additional_kwargs={}, response_metadata={})])

## 部分变量预填充：partial()

In [20]:
# 场景：为不同部门创建专用模板
base_template = ChatPromptTemplate.from_messages([
    ("system", "你是{department}的{role}"),
    ("user", "{task}")
])
# IT 部门
it_template = base_template.partial(
    department="IT 部门",
    role="技术支持"
)
# 销售部门
sales_template = base_template.partial(
    department="销售部门",
    role="销售顾问"
)
sales_template.invoke({"task": "为什么每年年底汽车会促销"})

ChatPromptValue(messages=[SystemMessage(content='你是销售部门的销售顾问', additional_kwargs={}, response_metadata={}), HumanMessage(content='为什么每年年底汽车会促销', additional_kwargs={}, response_metadata={})])

## 消息占位符

In [21]:
template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个有用的AI助手"),
        ("placeholder", "{conversation}"),
    ]
)
prompt_value = template.invoke(
    {
        "conversation": [
            ("human", "你好!"),
            ("ai", "今天我能帮你做什么？"),
            ("human", "你能给我做一个冰激凌吗？"),
            ("ai", "抱歉，我没有这样的能力"),
        ]
    }
)
print(prompt_value)

messages=[SystemMessage(content='你是一个有用的AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好!', additional_kwargs={}, response_metadata={}), AIMessage(content='今天我能帮你做什么？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你能给我做一个冰激凌吗？', additional_kwargs={}, response_metadata={}), AIMessage(content='抱歉，我没有这样的能力', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [26]:
template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个有用的AI助手"),
        MessagesPlaceholder("conversation")
    ]
)
prompt_value = template.invoke(
    {
        "conversation": [
            HumanMessage("你好！"),
            ("ai", "今天我能帮你做什么？"),
            ("human", "你能给我做一个冰激凌吗？"),
            ("ai", "抱歉，我没有这样的能力"),
        ]
    }
)
print(prompt_value)

messages=[SystemMessage(content='你是一个有用的AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好！', additional_kwargs={}, response_metadata={}), AIMessage(content='今天我能帮你做什么？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你能给我做一个冰激凌吗？', additional_kwargs={}, response_metadata={}), AIMessage(content='抱歉，我没有这样的能力', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
